# 04 — Transfermarkt Matching and Valuation Lookup

This notebook matches the FBref–Understat player-season dataset to Transfermarkt player records, then attaches market values around the observation season and two seasons later.

## Running this notebook
Run from the repository root or from the `notebooks/` directory. Place the source files in the locations shown in `data/README.md`. Install dependencies with `pip install -r requirements.txt`.

The matching thresholds, manual review decisions, case-study overrides and 180-day valuation tolerance are retained from the dissertation workflow. Review these decisions before reusing the pipeline on a different dataset.

# 04 — Transfermarkt Matching & Valuation Lookup

Matches the combined FBref+Understat dataset to the Kaggle Transfermarkt `players.csv` (for `player_id` and birth dates), then uses `player_valuations.csv` to attach:
- market value at the observation season (end of that season)
- market value N seasons later (for the value-growth composite signal)

**Matching approach:** candidates are blocked by both **estimated birth year** and **nationality** before fuzzy name scoring. Common names in high-supply football nations (e.g. Brazilian names) can still produce multiple same-nationality, same-birth-year-window candidates even after this — these `exact_ambiguous` cases are resolved separately using each candidate's actual club history from `player_valuations.csv`, cross-checked against the club your data already has them at.

In [1]:
import pandas as pd
import numpy as np
import html
import unicodedata
import pycountry
from rapidfuzz import process, fuzz, utils

from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)
PROCESSED_DIR.mkdir(exist_ok=True, parents=True)

## 1. Load data

`combined_df` is the output of notebook 03 (FBref + Understat, cleaned and matched).

In [2]:
combined_df = pd.read_csv(PROCESSED_DIR / "combined_fbref_understat_final.csv")
players_tm = pd.read_csv(RAW_DIR / "transfermarkt" / "players.csv")
valuations_tm = pd.read_csv(RAW_DIR / "transfermarkt" / "player_valuations.csv")

print(f"combined_df: {combined_df.shape}")
print(f"players_tm: {players_tm.shape}")
print(f"valuations_tm: {valuations_tm.shape}")

combined_df: (2872, 286)
players_tm: (50149, 26)
valuations_tm: (656301, 6)


In [3]:
# Check for duplicate player-season records after loading the source dataset.
dupe_check = combined_df.duplicated(subset=['Player', 'Season']).sum()
if dupe_check > 0:
    print(f"WARNING: {dupe_check} duplicate (Player, Season) rows on load — dropping.")
    combined_df = combined_df.drop_duplicates(subset=['Player', 'Season'], keep='first')

In [4]:
print(f"combined_df: {combined_df.shape}")

combined_df: (2872, 286)


## 2. Name normalization

In [5]:
def normalize_name(name):
    """Strip accents, decode HTML entities, lowercase, trim whitespace."""
    name = html.unescape(str(name))
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('utf-8')
    return name.strip().lower()

combined_df['player_key'] = combined_df['Player'].apply(normalize_name)

players_tm['player_key'] = players_tm['name'].apply(normalize_name)
players_tm['birth_year'] = pd.to_datetime(players_tm['date_of_birth'], errors='coerce').dt.year

## 3. Estimate each player's birth-year window from FBref Age + Season

FBref's `Age` is as of ~Feb 1 of the season's second year, so it can be off by a year either side of a player's true birth year. We take every season a player appears in and build a window wide enough to cover that uncertainty.

In [6]:
# NOTE: birth-year estimate uses season START year, not end year.
# This dataset's 'Age' field is recorded as of roughly the start of the
# season, not FBref's older Feb-1-of-second-year convention. Using the
# wrong reference point shifted estimates by ~1-2 years and caused real
# matches (e.g. Randal Kolo Muani, true birth year 1998, estimated as
# 2000) to fall just outside the window. Padding widened to ±2 as extra
# insurance given the convention isn't independently verified.
def season_start_year(season_str):
    """'2020-2021' -> 2020"""
    return int(str(season_str).split('-')[0])

combined_df['_season_start_year'] = combined_df['Season'].apply(season_start_year)
combined_df['_birth_year_estimate'] = combined_df['_season_start_year'] - combined_df['Age']

birth_year_windows = combined_df.groupby('player_key').agg(
    Player=('Player', 'first'),
    birth_year_min=('_birth_year_estimate', lambda x: x.min() - 2),
    birth_year_max=('_birth_year_estimate', lambda x: x.max() + 2),
).reset_index()

print(f"Unique players to match: {len(birth_year_windows)}")
birth_year_windows.head()

Unique players to match: 1539


,player_key,Player,birth_year_min,birth_year_max
0,aaron connolly,Aaron Connolly,1998.0,2002.0
1,aaron hickey,Aaron Hickey,2000.0,2004.0
2,aaron leya iseka,Aaron Leya Iseka,1996.0,2000.0
3,aaron martin,Aarón Martín,1995.0,1999.0
4,aaron ramsdale,Aaron Ramsdale,1996.0,2000.0


## 4. Nationality mapping (FIFA code → Transfermarkt country name)

In [7]:
def fifa_code_to_country_name(code):
    """Best-effort map from a 3-letter nationality code to a country name
    matching Transfermarkt's 'country_of_citizenship' format."""
    overrides = {
        'ENG': 'England', 'SCO': 'Scotland', 'WAL': 'Wales', 'NIR': 'Northern Ireland',
        'NED': 'Netherlands', 'GER': 'Germany', 'POR': 'Portugal', 'ESP': 'Spain',
        'FRA': 'France', 'BRA': 'Brazil', 'ARG': 'Argentina', 'CIV': 'Ivory Coast',
        'KVX': 'Kosovo', 'RSA': 'South Africa', 'KOR': 'South Korea',
    }
    if code in overrides:
        return overrides[code]
    try:
        country = pycountry.countries.get(alpha_3=code)
        return country.name if country else None
    except Exception:
        return None

combined_df['_country_name'] = combined_df['Nation'].apply(fifa_code_to_country_name)

nation_by_player = combined_df.groupby('player_key')['_country_name'].agg(
    lambda x: x.mode().iloc[0] if not x.mode().empty else None
).reset_index().rename(columns={'_country_name': 'country_name'})

birth_year_windows = birth_year_windows.merge(nation_by_player, on='player_key', how='left')

unmapped = birth_year_windows[birth_year_windows['country_name'].isna()]
print(f"Players with unmapped nationality: {len(unmapped)}")
if len(unmapped):
    print("These will fall back to birth-year-only blocking — check if a code needs adding to 'overrides' above:")
    print(unmapped[['Player']].head(10))

Players with unmapped nationality: 148
These will fall back to birth-year-only blocking — check if a code needs adding to 'overrides' above:
                  Player
14          Ablie Jallow
21            Adam Ounas
32         Agustín Rogel
84        Amadou Diawara
89          Amine Gouiri
96      Anderson Lucoqui
97           Andi Zeqiri
108        Andreas Bruus
109  Andreas Christensen
111   Andreas Skov Olsen


## 5. Match — birth year + nationality blocking, then exact name, then fuzzy fallback

Three tiers: auto-accept (score ≥ 90), manual review (85–89), unmatched (< 85). Exact name matches with **multiple** same-nationality, same-birth-year-window candidates are tagged `exact_ambiguous` and resolved separately in Section 6.

In [8]:
HIGH_THRESHOLD = 90
REVIEW_THRESHOLD = 85

auto_rows, review_rows, unmatched_rows = [], [], []

for _, row in birth_year_windows.iterrows():
    candidates = players_tm[
        (players_tm['birth_year'] >= row['birth_year_min']) &
        (players_tm['birth_year'] <= row['birth_year_max'])
    ]

    nationality_applied = False
    if row['country_name'] is not None:
        nat_filtered = candidates[candidates['country_of_citizenship'] == row['country_name']]
        if not nat_filtered.empty:
            candidates = nat_filtered
            nationality_applied = True

    if candidates.empty:
        unmatched_rows.append({**row.to_dict(), 'reason': 'no candidates after blocking'})
        continue

    exact = candidates[candidates['player_key'] == row['player_key']]
    if len(exact) == 1:
        auto_rows.append({**row.to_dict(),
                           'player_id': exact.iloc[0]['player_id'],
                           'matched_name': exact.iloc[0]['name'],
                           'match_score': 100,
                           'match_type': 'exact',
                           'nationality_applied': nationality_applied})
        continue
    elif len(exact) > 1:
        for _, cand in exact.iterrows():
            review_rows.append({**row.to_dict(),
                                 'player_id': cand['player_id'],
                                 'matched_name': cand['name'],
                                 'match_score': 100,
                                 'match_type': 'exact_ambiguous',
                                 'nationality_applied': nationality_applied})
        continue

    candidate_names = candidates['player_key'].tolist()
    best = process.extractOne(row['player_key'], candidate_names,
                               scorer=fuzz.token_sort_ratio, processor=utils.default_process)

    if best is None:
        unmatched_rows.append({**row.to_dict(), 'reason': 'no fuzzy candidate'})
        continue

    matched_key, score, idx = best
    matched_candidate = candidates.iloc[idx]
    result = {**row.to_dict(),
              'player_id': matched_candidate['player_id'],
              'matched_name': matched_candidate['name'],
              'match_score': score,
              'match_type': 'fuzzy',
              'nationality_applied': nationality_applied}

    if score >= HIGH_THRESHOLD:
        auto_rows.append(result)
    elif score >= REVIEW_THRESHOLD:
        review_rows.append(result)
    else:
        unmatched_rows.append({**row.to_dict(), 'reason': f'low score ({score})'})

auto_matches = pd.DataFrame(auto_rows)
review_matches = pd.DataFrame(review_rows)
still_unmatched = pd.DataFrame(unmatched_rows)

print(f"Auto-matched (exact, unambiguous): {len(auto_matches)}")
print(f"Review tier (fuzzy + exact_ambiguous): {len(review_matches)}")
print(f"  - of which exact_ambiguous: {(review_matches['match_type'] == 'exact_ambiguous').sum() if len(review_matches) else 0}")
print(f"  - of which fuzzy: {(review_matches['match_type'] == 'fuzzy').sum() if len(review_matches) else 0}")
print(f"Unmatched: {len(still_unmatched)}")

Auto-matched (exact, unambiguous): 1451
Review tier (fuzzy + exact_ambiguous): 37
  - of which exact_ambiguous: 28
  - of which fuzzy: 9
Unmatched: 67


### Check your named case-study players directly

Palmer, Olise, Rogers, and Gyökeres matter more than average here — verify each matched correctly rather than trusting they're buried correctly in the review table.

In [9]:
case_study_players = ['Cole Palmer', 'Michael Olise', 'Randal Kolo Muani']
# Note: Morgan Rogers and Viktor Gyökeres were originally planned case studies
# but are excluded here — Rogers fell just under the 900-minute threshold
# (638 mins, Aston Villa 2023-24) and Gyökeres never played in a Big 5 league
# during his 18-24 window (breakout occurred at Sporting CP, outside scope).
# Both are discussed in the methodology as a scope limitation rather than
# given full case-study treatment.

all_matches = pd.concat([auto_matches, review_matches], ignore_index=True)

for name in case_study_players:
    key = normalize_name(name)
    result = all_matches[all_matches['player_key'] == key]
    print(f"\n{name}:")
    if not result.empty:
        print(result[['matched_name', 'match_score', 'match_type']].to_string(index=False))
    else:
        print("  NOT FOUND — look this one up manually in players_tm before proceeding")


Cole Palmer:
matched_name  match_score match_type
 Cole Palmer        100.0      exact

Michael Olise:
 matched_name  match_score match_type
Michael Olise        100.0      exact

Randal Kolo Muani:
     matched_name  match_score match_type
Randal Kolo Muani        100.0      exact


## 6. Resolve `exact_ambiguous` matches using club history

Common names in high-supply football nations (Brazilian names especially) can share name, nationality, *and* an overlapping birth-year window across multiple real players. `player_id` doesn't help here — it only identifies who's who on Transfermarkt's side, it can't tell you which of several already-distinct IDs belongs to your row.

Instead, each candidate's actual club history (`player_valuations.csv`, dated) is checked against the club your FBref/Understat data already has that player at. Only one candidate should show a genuine overlap in most cases.

In [10]:
ambiguous = review_matches[review_matches['match_type'] == 'exact_ambiguous'].copy()
print(f"Ambiguous name-groups to resolve: {ambiguous['player_key'].nunique()}")
print(f"Total candidate rows: {len(ambiguous)}")

resolved_rows = []
still_ambiguous_keys = []

for player_key, group in ambiguous.groupby('player_key'):
    known_teams = set(
        combined_df.loc[combined_df['player_key'] == player_key, 'Team']
        .dropna().str.lower().str.strip()
    )

    matches_found = []
    for _, cand in group.iterrows():
        cand_id = cand['player_id']
        cand_clubs = set(
            valuations_tm.loc[valuations_tm['player_id'] == cand_id, 'current_club_name']
            .dropna().str.lower().str.strip()
        )
        overlap = any(kt in cc or cc in kt for kt in known_teams for cc in cand_clubs)
        if overlap:
            matches_found.append(cand)

    if len(matches_found) == 1:
        resolved_rows.append(matches_found[0])
    else:
        # 0 matches (data too sparse) or 2+ (genuinely still ambiguous) — flag for manual lookup
        still_ambiguous_keys.append(player_key)

resolved_ambiguous = pd.DataFrame(resolved_rows)

print(f"\nResolved automatically via club history: {len(resolved_ambiguous)}")
print(f"Still ambiguous (needs manual lookup): {len(still_ambiguous_keys)}")
if still_ambiguous_keys:
    still_ambiguous_names = ambiguous[ambiguous['player_key'].isin(still_ambiguous_keys)]['Player'].unique()
    print(list(still_ambiguous_names))

Ambiguous name-groups to resolve: 12
Total candidate rows: 28

Resolved automatically via club history: 9
Still ambiguous (needs manual lookup): 3
['Antony', 'Danilo', 'Vitinha']


For any names still ambiguous after this, look up the correct `player_id` manually (search the player name on Transfermarkt directly) and add it here:

In [11]:
# Manually resolved player_ids for anything club-history couldn't disambiguate.
# Format: 'normalized player_key': player_id
manual_ambiguous_resolutions = {
    # 'joão victor': 738504,
}

manual_rows = []
for player_key, pid in manual_ambiguous_resolutions.items():
    cand = ambiguous[(ambiguous['player_key'] == player_key) & (ambiguous['player_id'] == pid)]
    if not cand.empty:
        manual_rows.append(cand.iloc[0])

manually_resolved = pd.DataFrame(manual_rows)
print(f"Manually resolved: {len(manually_resolved)}")

Manually resolved: 0


## 7. Eyeball the fuzzy review tier before accepting

(`exact_ambiguous` rows are excluded here — already handled in Section 6.)

In [12]:
pd.set_option('display.max_rows', None)
fuzzy_review = review_matches[review_matches['match_type'] == 'fuzzy']
fuzzy_review[['Player', 'matched_name', 'match_score', 'match_type',
              'nationality_applied', 'birth_year_min', 'birth_year_max']].sort_values('match_score', ascending=False)

,Player,matched_name,match_score,match_type,nationality_applied,birth_year_min,birth_year_max
19,Nicolás González,Nico González,89.655172,fuzzy,True,1996.0,2004.0
20,Oliver McBurnie,Oli McBurnie,88.888889,fuzzy,True,1994.0,1998.0
11,Illia Zabarnyi,Ilya Zabarnyi,88.888889,fuzzy,True,2001.0,2005.0
24,Salvador Ferrer,Salva Ferrer,88.888889,fuzzy,True,1996.0,2000.0
6,David Costa,David Da Costa,88.000000,fuzzy,True,1999.0,2003.0
12,Jan-Niklas Beste,Niklas Beste,85.714286,fuzzy,True,1997.0,2001.0
21,Đorđe Petrović,Djordje Petrovic,85.714286,fuzzy,True,1998.0,2002.0
27,Timmy Tillman,Timothy Tillman,85.714286,fuzzy,True,1997.0,2001.0
28,Valentino Livramento,Tino Livramento,85.714286,fuzzy,True,2001.0,2005.0


In [13]:
# After manually checking the fuzzy review table above, list the pairs you're
# confident about (or the ones to discard — whichever is the shorter list).
confirmed_pairs = [('Nicolás González',	'Nico González'),
                   ('Illia Zabarnyi','Ilya Zabarnyi'),
                   ('Salvador Ferrer','Salva Ferrer'),
                   ('Oliver McBurnie','Oli McBurnie'),
                   ('Jan-Niklas Beste',	'Niklas Beste'),
                   ('Valentino Livramento',	'Tino Livramento'),
    # ('Player Name', 'Matched Transfermarkt Name'),
]

confirmed_fuzzy = fuzzy_review[
    fuzzy_review.apply(lambda r: (r['Player'], r['matched_name']) in confirmed_pairs, axis=1)
]

print(f"Confirmed fuzzy matches: {len(confirmed_fuzzy)} / {len(fuzzy_review)}")

Confirmed fuzzy matches: 6 / 9


## 8. Combine everything into the final player_id lookup

In [14]:
player_id_lookup = pd.concat(
    [auto_matches, resolved_ambiguous, manually_resolved, confirmed_fuzzy],
    ignore_index=True
)[['player_key', 'player_id', 'match_score', 'match_type']]

# Safety check: make sure no player_key ended up with more than one player_id
dupe_check = player_id_lookup.groupby('player_key')['player_id'].nunique()
still_conflicting = dupe_check[dupe_check > 1]
if len(still_conflicting):
    print(f"WARNING: {len(still_conflicting)} players still have multiple player_ids assigned — resolve before continuing:")
    print(still_conflicting)
else:
    print("No conflicts — every matched player has exactly one player_id.")

print(f"\nTotal players with a resolved Transfermarkt ID: {len(player_id_lookup)}")

# Save your manual review decisions so you don't have to redo this if you re-run the pipeline
import json
with open(str(PROCESSED_DIR / "confirmed_transfermarkt_matches.json"), "w") as f:
    json.dump({
        'confirmed_fuzzy_pairs': confirmed_pairs,
        'manual_ambiguous_resolutions': manual_ambiguous_resolutions,
    }, f, indent=2)

No conflicts — every matched player has exactly one player_id.

Total players with a resolved Transfermarkt ID: 1466


In [15]:
# Manual safety overrides for case-study players — verified directly
# against Transfermarkt rather than relying solely on automated matching,
# given how central these three are to the dissertation's evaluation.
case_study_overrides = {
    'michael olise': 566723,
    'randal kolo muani': 487969,
}

override_rows = []
for player_key, pid in case_study_overrides.items():
    override_rows.append({
        'player_key': player_key,
        'player_id': pid,
        'match_score': 100,
        'match_type': 'manual_override',
    })

case_study_df = pd.DataFrame(override_rows)

# Drop any existing entry for these players first (whether from automated
# matching OR a previous run of this same cell) so the override always wins
# and this cell is safe to run more than once without creating duplicates.
player_id_lookup = player_id_lookup[~player_id_lookup['player_key'].isin(case_study_overrides.keys())]
player_id_lookup = pd.concat([player_id_lookup, case_study_df], ignore_index=True)
player_id_lookup = player_id_lookup.drop_duplicates(subset=['player_key'], keep='last')

print(f"Total after case-study overrides: {len(player_id_lookup)}")

Total after case-study overrides: 1466


## 9. Attach `player_id` to every row of the combined dataset

In [16]:
combined_df = combined_df.merge(player_id_lookup, on='player_key', how='left')

matched_pct = combined_df['player_id'].notna().mean()
print(f"Player-season rows with a Transfermarkt ID: {matched_pct:.1%}")

Player-season rows with a Transfermarkt ID: 95.8%


## 10. Valuation lookup — nearest value at observation date and N seasons later

Transfermarkt valuations aren't recorded on a fixed schedule, so `pd.merge_asof` finds the **nearest available valuation** to each target date, within a tolerance window (default 180 days) so a stale value isn't silently attached.

In [17]:
N_SEASONS_AHEAD = 2       # how far ahead to check for the value-growth signal
TOLERANCE_DAYS = 180      # max gap allowed between target date and nearest actual valuation

valuations_tm['date'] = pd.to_datetime(valuations_tm['date'], errors='coerce')
valuations_tm = valuations_tm.dropna(subset=['date', 'player_id']).sort_values('date')

def season_end_date(season_str, years_ahead=0):
    """'2022-2023' -> Timestamp('2023-06-30'), optionally shifted N seasons ahead."""
    end_year = int(str(season_str).split('-')[-1]) + years_ahead
    return pd.Timestamp(year=end_year, month=6, day=30)

def nearest_valuations(targets, date_col, tolerance_days=TOLERANCE_DAYS):
    """
    targets: DataFrame with 'player_id' and a datetime column named date_col.
    Returns targets merged with the nearest valuation per player_id within tolerance.
    """
    targets = targets.dropna(subset=['player_id', date_col]).copy()
    targets['player_id'] = targets['player_id'].astype(valuations_tm['player_id'].dtype)
    targets = targets.sort_values(date_col)

    merged = pd.merge_asof(
        targets,
        valuations_tm[['player_id', 'date', 'market_value_in_eur', 'current_club_name']],
        left_on=date_col,
        right_on='date',
        by='player_id',
        direction='nearest',
        tolerance=pd.Timedelta(days=tolerance_days),
    )
    return merged

In [18]:
# Value at observation season
obs_targets = combined_df[['Player', 'Season', 'player_id']].dropna(subset=['player_id']).copy()
obs_targets['_target_date'] = obs_targets['Season'].apply(season_end_date)

obs_result = nearest_valuations(obs_targets, '_target_date')
obs_result = obs_result.rename(columns={
    'market_value_in_eur': 'value_at_observation',
    'current_club_name': 'club_at_observation',
    'date': 'observation_valuation_date',
})

print(f"Rows with a value found within {TOLERANCE_DAYS} days of observation: {obs_result['value_at_observation'].notna().sum()} / {len(obs_result)}")

Rows with a value found within 180 days of observation: 2750 / 2751


In [19]:
# Value N seasons later (for the value-growth composite signal)
future_targets = combined_df[['Player', 'Season', 'player_id']].dropna(subset=['player_id']).copy()
future_targets['_target_date'] = future_targets['Season'].apply(season_end_date, years_ahead=N_SEASONS_AHEAD)

future_result = nearest_valuations(future_targets, '_target_date')
future_result = future_result.rename(columns={
    'market_value_in_eur': f'value_{N_SEASONS_AHEAD}yr_later',
    'current_club_name': f'club_{N_SEASONS_AHEAD}yr_later',
    'date': 'future_valuation_date',
})

print(f"Rows with a future value found within {TOLERANCE_DAYS} days: {future_result[f'value_{N_SEASONS_AHEAD}yr_later'].notna().sum()} / {len(future_result)}")
print("Note: rows from your most recent seasons won't have a future value yet — this is expected (censored data), not an error.")

Rows with a future value found within 180 days: 2646 / 2751
Note: rows from your most recent seasons won't have a future value yet — this is expected (censored data), not an error.


## 11. Bring it all together

In [20]:
final_df = combined_df.merge(
    obs_result[['Player', 'Season', 'player_id', 'value_at_observation', 'club_at_observation']],
    on=['Player', 'Season', 'player_id'], how='left'
).merge(
    future_result[['Player', 'Season', 'player_id', f'value_{N_SEASONS_AHEAD}yr_later', f'club_{N_SEASONS_AHEAD}yr_later']],
    on=['Player', 'Season', 'player_id'], how='left'
)

final_df.to_csv(PROCESSED_DIR / "combined_with_valuations.csv", index=False)
print(f"Final dataset: {final_df.shape}")
final_df.head()

Final dataset: (2872, 296)


,Comp,season,Team,Player,Nation,Pos,Age,born_,MP,Playing Time_Starts,Min,90s,Gls,Ast,G+A,Performance_G-PK,Performance_PK,Performance_PKatt,Performance_CrdY,Performance_CrdR,Expected_xG,Expected_npxG,Expected_xAG,Expected_npxG+xAG,Progression_PrgC,Progression_PrgP,Progression_PrgR,Per 90 Minutes_Gls,Per 90 Minutes_Ast,Per 90 Minutes_G+A,Per 90 Minutes_G-PK,Per 90 Minutes_G+A-PK,Per 90 Minutes_xG,Per 90 Minutes_xAG,Per 90 Minutes_xG+xAG,Per 90 Minutes_npxG,Per 90 Minutes_npxG+xAG,90s_,Expected_G-xG,Expected_np:G-xG,...,Age_stats_misc,Born_stats_misc,90s_stats_misc,CrdY_stats_misc,CrdR_stats_misc,2CrdY,Fls,Fld,Off,Crs,OG,player_key,player,team,min,NPG,xG,NPxG,xA,xGChain,xGBuildup,xG90,NPxG90,xA90,xG90xA90,NPxG90xA90,xGChain90,xGBuildup90,Season_understat,league,_season_start_year,_birth_year_estimate,_country_name,player_id,match_score,match_type,value_at_observation,club_at_observation,value_2yr_later,club_2yr_later
0,ENG-Premier League,2223.0,Arsenal,Aaron Ramsdale,ENG,GK,24.0,1998.0,38,38.0,3420,38.0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,aaron ramsdale,Aaron Ramsdale,Arsenal,3420,0,0.0,0.0,0.00,5.10,5.10,0.0,0.0,0.0,0.0,0.0,0.13,0.13,2022 - 2023,Premier League,2022,1998.0,England,427568.0,100.0,exact,40000000.0,Arsenal FC,16000000.0,Southampton FC
1,FRA-Ligue 1,2122.0,Monaco,Alexander Nübel,GER,GK,24.0,1996.0,38,38.0,3420,38.0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,alexander nubel,Alexander Nübel,Monaco,3420,0,0.0,0.0,0.00,5.34,5.34,0.0,0.0,0.0,0.0,0.0,0.14,0.14,2021 - 2022,Ligue 1,2021,1997.0,Germany,195778.0,100.0,exact,8000000.0,AS Monaco,12000000.0,VfB Stuttgart
2,ESP-La Liga,2223.0,Valencia,Giorgi Mamardashvili,GEO,GK,21.0,2000.0,38,38.0,3420,38.0,0,0,0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,giorgi mamardashvili,Giorgi Mamardashvili,Valencia,3420,0,0.0,0.0,0.00,3.62,3.62,0.0,0.0,0.0,0.0,0.0,0.10,0.10,2022 - 2023,La Liga,2022,2001.0,Georgia,502676.0,100.0,exact,25000000.0,Valencia CF,30000000.0,Valencia CF
3,ITA-Serie A,2122.0,Empoli,Guglielmo Vicario,ITA,GK,24.0,1996.0,38,38.0,3420,38.0,0,0,0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,guglielmo vicario,Guglielmo Vicario,Empoli,3420,0,0.0,0.0,0.01,6.05,6.04,0.0,0.0,0.0,0.0,0.0,0.16,0.16,2021 - 2022,Serie A,2021,1997.0,Italy,286047.0,100.0,exact,10000000.0,FC Empoli,35000000.0,Tottenham Hotspur
4,FRA-Ligue 1,2122.0,Nantes,Alban Lafont,CIV,GK,22.0,1999.0,38,38.0,3420,38.0,0,0,0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,38.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,alban lafont,Alban Lafont,Nantes,3420,0,0.0,0.0,0.00,2.01,2.01,0.0,0.0,0.0,0.0,0.0,0.05,0.05,2021 - 2022,Ligue 1,2021,1999.0,Ivory Coast,357117.0,100.0,exact,16000000.0,FC Nantes,9000000.0,FC Nantes


## 12. Match-rate summary for your data section

In [21]:
total_players = len(birth_year_windows)
exact_n = (player_id_lookup['match_type'] == 'exact').sum()
ambiguous_resolved_n = (player_id_lookup['match_type'] == 'exact_ambiguous').sum()
fuzzy_n = (player_id_lookup['match_type'] == 'fuzzy').sum()
unmatched_n = total_players - player_id_lookup['player_key'].nunique()

print(f"Total unique players: {total_players}")
print(f"Exact, unambiguous matches: {exact_n} ({exact_n/total_players:.1%})")
print(f"Exact but ambiguous, resolved via club history/manual lookup: {ambiguous_resolved_n} ({ambiguous_resolved_n/total_players:.1%})")
print(f"Fuzzy matches confirmed: {fuzzy_n} ({fuzzy_n/total_players:.1%})")
print(f"Unmatched: {unmatched_n} ({unmatched_n/total_players:.1%})")
print()
print(f"Player-season rows with observation-point value: {final_df['value_at_observation'].notna().mean():.1%}")
print(f"Player-season rows with {N_SEASONS_AHEAD}yr-later value: {final_df[f'value_{N_SEASONS_AHEAD}yr_later'].notna().mean():.1%} (many recent seasons expected to be missing — censored, not an error)")

Total unique players: 1539
Exact, unambiguous matches: 1421 (92.3%)
Exact but ambiguous, resolved via club history/manual lookup: 9 (0.6%)
Fuzzy matches confirmed: 34 (2.2%)
Unmatched: 73 (4.7%)

Player-season rows with observation-point value: 95.8%
Player-season rows with 2yr-later value: 92.1% (many recent seasons expected to be missing — censored, not an error)
